# Metric Correlation Analysis: Classical vs. LLM-based Metrics (EXP1-35)

This notebook analyzes the correlation between classical XAI metrics (Fidelity, Stability, Sparsity) and LLM-based evaluation scores (Coherence, Faithfulness, Usefulness) from Experiment 1 (Adult Dataset).

## Objectives
1.  **Exploratory Data Analysis**: Visualize distributions of all metrics.
2.  **Correlation Analysis**: Compute Pearson/Spearman correlations between classical and LLM score.
3.  **Hypothesis Testing**: Determine if correlations differ significantly by Model or Explainer type.

## Data Sources
- **LLM Scores**: `experiments/exp1_adult/llm_eval/results_full.json`
- **Classical Metrics**: `experiments/exp1_adult/results/*/metrics.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import json
import os
import glob

# Set visualization settings
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

## 1. Data Loading

In [ ]:
# 1.1 Load LLM Evaluation Results
llm_results_path = "../../experiments/exp1_adult/llm_eval/results_full.json"

with open(llm_results_path, 'r') as f:
    llm_data = json.load(f)

# Flatten the nested 'eval_scores' dictionary
flat_llm_data = []
for record in llm_data:
    flat_record = {
        'instance_id': record['instance_id'],
        'model': record['model'],
        'explainer': record['explainer'],
        'quadrant': record['quadrant'],
        'llm_coherence': record['eval_scores']['coherence'],
        'llm_faithfulness': record['eval_scores']['faithfulness'],
        'llm_usefulness': record['eval_scores']['usefulness']
    }
    flat_llm_data.append(flat_record)

df_llm = pd.DataFrame(flat_llm_data)
print(f"LLM Data Shape: {df_llm.shape}")
df_llm.head()

In [ ]:
# 1.2 Load Classical Metrics
metrics_files = glob.glob("../../experiments/exp1_adult/results/*/metrics.csv")
print(f"Found {len(metrics_files)} metrics files: {metrics_files}")

df_metrics_list = []

for file_path in metrics_files:
    # Extract model and explainer from path (e.g., .../rf_lime/metrics.csv)
    dir_name = os.path.basename(os.path.dirname(file_path))
    # Assuming directory naming convention like 'rf_lime', 'xgb_shap'
    if '_' in dir_name:
        model_prefix, explainer_suffix = dir_name.split('_', 1)
        # Normalize names to match LLM data (rf -> rf/random_forest?, let's check)
        # LLM data has 'xgboost', 'rf'
        model = 'xgboost' if model_prefix == 'xgb' else model_prefix
        explainer = explainer_suffix
    else:
        continue # Skip folders that don't match pattern

    df_temp = pd.read_csv(file_path)
    df_temp['model'] = model
    df_temp['explainer'] = explainer
    
    # Standardize instance_id type
    df_temp['instance_id'] = df_temp['instance_id'].astype(int)
    
    df_metrics_list.append(df_temp)

df_classical = pd.concat(df_metrics_list, ignore_index=True)
print(f"Classical Metrics Data Shape: {df_classical.shape}")
df_classical.head()

In [ ]:
# 1.3 Merge Datasets
# Join on instance_id, model, and explainer
df_merged = pd.merge(
    df_classical, 
    df_llm, 
    on=['instance_id', 'model', 'explainer', 'quadrant'], 
    how='inner'
)

print(f"Merged Data Shape: {df_merged.shape}")

# Check for consistency
if df_merged.shape[0] < df_llm.shape[0]:
    print(f"WARNING: Lost {df_llm.shape[0] - df_merged.shape[0]} LLM records during merge.")
    
df_merged.head()

## 2. Exploratory Data Analysis

In [ ]:
# 2.1 LLM Score Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df_merged, x='llm_coherence', ax=axes[0], palette='viridis')
axes[0].set_title('Distribution of Coherence Scores')

sns.countplot(data=df_merged, x='llm_faithfulness', ax=axes[1], palette='viridis')
axes[1].set_title('Distribution of Faithfulness Scores')

sns.countplot(data=df_merged, x='llm_usefulness', ax=axes[2], palette='viridis')
axes[2].set_title('Distribution of Usefulness Scores')

plt.tight_layout()
plt.show()

In [ ]:
# 2.2 Classical Metrics Distributions (Boxplots by Explainer)
metrics_to_plot = ['metric_fidelity', 'metric_stability', 'metric_sparsity', 'metric_cost']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    if metric in df_merged.columns:
        sns.boxplot(data=df_merged, x='explainer', y=metric, hue='model', ax=axes[i])
        axes[i].set_title(f'{metric} by Explainer & Model')
    else:
        axes[i].text(0.5, 0.5, f'{metric} not found', ha='center')

plt.tight_layout()
plt.show()

## 3. Correlation Analysis

In [ ]:
# 3.1 Overall Correlation Matrix
# Select numerical columns for correlation
classical_cols = [c for c in df_merged.columns if c.startswith('metric_')]
llm_cols = ['llm_coherence', 'llm_faithfulness', 'llm_usefulness']

corr_data = df_merged[classical_cols + llm_cols].corr(method='spearman')

plt.figure(figsize=(12, 10))
sns.heatmap(corr_data, annot=True, cmap='RdBu_r', center=0, fmt='.2f', vmin=-1, vmax=1)
plt.title('Spearman Correlation: Classical Metrics vs. LLM Scores')
plt.show()

In [ ]:
# 3.2 Focused Correlation Heatmap (Classical vs LLM only)
plt.figure(figsize=(8, 6))
subset_corr = corr_data.loc[classical_cols, llm_cols]
sns.heatmap(subset_corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation: Classical Metrics (Rows) vs. LLM Scores (Cols)')
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Stratified Analysis: By Explainer
explainers = df_merged['explainer'].unique()

fig, axes = plt.subplots(1, len(explainers), figsize=(18, 6), sharey=True)

for i, explainer in enumerate(explainers):
    subset = df_merged[df_merged['explainer'] == explainer]
    corr = subset[classical_cols + llm_cols].corr(method='spearman')
    focus_corr = corr.loc[classical_cols, llm_cols]
    
    sns.heatmap(focus_corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=axes[i], cbar=i==len(explainers)-1, vmin=-1, vmax=1)
    axes[i].set_title(f'Explainer: {explainer}')

plt.tight_layout()
plt.show()

## 4. Key Relationships (Scatter Plots)

In [ ]:
def plot_relationship(x_metric, y_score, title=None):
    plt.figure(figsize=(8, 6))
    sns.regplot(data=df_merged, x=x_metric, y=y_score, x_jitter=0.01, y_jitter=0.1, scatter_kws={'alpha':0.5}, line_kws={'color': 'red'})
    sns.scatterplot(data=df_merged, x=x_metric, y=y_score, hue='explainer', style='model')
    plt.title(title if title else f'{x_metric} vs {y_score}')
    plt.show()

# 1. Faithfulness Gap vs LLM Faithfulness
if 'metric_faithfulness_gap' in df_merged.columns:
    plot_relationship('metric_faithfulness_gap', 'llm_faithfulness')

# 2. Fidelity vs LLM Faithfulness
plot_relationship('metric_fidelity', 'llm_faithfulness')

# 3. Stability vs LLM Coherence/Stability
plot_relationship('metric_stability', 'llm_coherence')

## 5. Statistical Tests

In [ ]:
# Calculate P-values for correlations
from scipy.stats import spearmanr

results = []
for c_metric in classical_cols:
    for l_score in llm_cols:
        r, p = spearmanr(df_merged[c_metric], df_merged[l_score])
        results.append({
            'Classical Metric': c_metric,
            'LLM Score': l_score,
            'Spearman R': r,
            'P-Value': p,
            'Significant': p < 0.05
        })

df_stats = pd.DataFrame(results)
print("Significant Correlations:")
df_stats[df_stats['Significant']].sort_values('P-Value')

## 6. Conclusions

Based on the analysis above:

-   **Fidelity vs Faithfulness**: [Observation to be filled]
-   **Stability vs Coherence**: [Observation to be filled]
-   **Model Differences**: [Observation to be filled]